In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score , mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
rest_path = os.path.join(path, 'Q1_data.csv')
df_rest = pd.read_csv(rest_path)

print(f"Dataset shape: {df_rest.shape}")

In [ ]:
# Task 2: Write your code here:
df_rest.head()

In [ ]:
# Task 3: Write your code here:
df_rest.info()

In [ ]:
# Task 4: Write your code here:
df_rest.describe()

In [ ]:
# Task 5: Write your code here:

# delivery_time (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_rest['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Plot')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_rest= df_rest.drop(columns=['Order_ID'], axis=1)

df_rest.info()

In [ ]:
# Task 2: Write your code here:

# the columns that has missing values :
# 1. Weather 2. Traffic_Level 3. Time_of_Day 4. Courier_Experience_yrs 5. Delivery_Time


# Fill with mode (categorical) 1. Weather 2. Traffic_Level 3. Time_of_Day

print("\n\n --------  Fill with mode:   --------\n")

df_mode = df_rest.copy()
df_mode['Weather'] = df_mode['Weather'].fillna(df_mode['Weather'].mode()[0])
df_mode['Traffic_Level'] = df_mode['Traffic_Level'].fillna(df_mode['Traffic_Level'].mode()[0])
df_mode['Time_of_Day'] = df_mode['Time_of_Day'].fillna(df_mode['Time_of_Day'].mode()[0])
print(df_mode)


# Fill with mean/median (numerical) 1. Courier_Experience_yrs 2. Delivery_Time

print("\n\n --------  Fill with mean/median:   --------\n")

df_mean = df_mode.copy()
df_mean['Delivery_Time'] = df_mean['Delivery_Time'].fillna(df_mean['Delivery_Time'].mean())
df_mean['Courier_Experience_yrs'] = df_mean['Courier_Experience_yrs'].fillna(df_mean['Courier_Experience_yrs'].median())
print(df_mean)

df_clean = df_mean.copy()

print("\n\n --------  Final clean dataset check up   --------\n")
df_clean.info()

In [ ]:
# Task 3: Write your code here:

df = df_clean.drop_duplicates()

df.info()


In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Categorical Columns: ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df


In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)

scaler = StandardScaler()
scaled = scaler.fit_transform(df)

print("Original:\n", df)
print("\nScaled (mean ≈ 0, std ≈ 1):\n", scaled)
print("\nMeans:", scaled.mean(axis=0))
print("Stds:", scaled.std(axis=0))

df.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()


check_target_imbalance(df, "Delivery_Time")
print("\nTarget is imbalnce as we observe from the Histogram")

In [ ]:
# Task 1: Write your code here:

X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# # Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

# model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# model.fit(X_train, y_train)
# print("Model trained!")

# kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# mae_scores = []

# for train_idx, val_idx in kfold.split(X_train):
#     X_train, X_test = X_train[train_idx], X_train[val_idx]
#     y_train, y_pred = y_train.iloc[train_idx], y_train.iloc[val_idx]

#     # Train and predict
#     model.fit(X_train, y_train)
#     y_fold_pred = model.predict(X_test)

#     # Calculate metrics
#     mae_scores.append(mean_absolute_error(y_test, y_pred))

# mae_scores = np.array(mae_scores)

# print(f"5-Fold CV Results:")
# print(f"MAE:  ${mae_scores.mean():,.2f}")


kf = KFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Evaluate using MAE (Mean Absolute Error) ONLY
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

# Print the averaged score across all folds

In [ ]:
# Task 1: Write your code here:

feature_cols = ['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# delivery_time (target variable)
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Delivery Time Plot')
plt.xlabel('time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: